# Federated Transfer Learning (FTL) with Additive Homomorphic Encryption

This notebook demonstrates:

1. Client 1 trains a model on Dataset A
2. Model weights are encrypted using **Paillier Additive Homomorphic Encryption**
3. Client 2 receives encrypted parameters
4. Secure aggregation / update occurs without revealing raw data
5. Model is decrypted and fine-tuned on Client 2
6. Compare accuracy with isolated training


**Implementation steps**

1. Dataset loading (Breast Cancer dataset)
2. Client split
Client 1 → Dataset A
Client 2 → Dataset B
3. Baseline isolated training
4. Client 1 pre-training
5. Additive Homomorphic Encryption
- Paillier key generation
- Encryption of model weights
6. Encrypted aggregation
7. Decryption of updated weights
8. Fine-tuning on Client 2
9. Accuracy comparison

**Core Privacy Technique Used**

Additive Homomorphic Encryption
Property:

𝐸𝑛𝑐(𝑎)+𝐸𝑛𝑐(𝑏)= 𝐸𝑛𝑐(𝑎+𝑏)

This allows computation on encrypted model parameters without revealing the data.

## Install Required Library

In [1]:
!pip install phe

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.7/53.7 kB 2.2 MB/s eta 0:00:00


## Import Libraries

In [2]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from phe import paillier

## Load Dataset and Simulate Two Clients

In [3]:
data = load_breast_cancer()
X = data.data
y = data.target

X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42)

split = int(0.5 * len(X_train))

XA = X_train[:split]
yA = y_train[:split]

XB = X_train[split:]
yB = y_train[split:]

print('Client 1 size:', XA.shape)
print('Client 2 size:', XB.shape)

Client 1 size: (227, 30)
Client 2 size: (228, 30)


## Baseline: Isolated Training (Client 2 Only)

In [4]:
iso_model = LogisticRegression(max_iter=500)
iso_model.fit(XB,yB)

pred_iso = iso_model.predict(X_test)
acc_iso = accuracy_score(y_test,pred_iso)

print('Isolated Training Accuracy:', acc_iso)

Isolated Training Accuracy: 0.956140350877193


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


## Client 1 Pre-training

In [5]:
modelA = LogisticRegression(max_iter=500)
modelA.fit(XA,yA)

weightsA = modelA.coef_[0]
biasA = modelA.intercept_[0]

print('Client 1 training completed')

Client 1 training completed


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


## Generate Paillier Keys

In [6]:
public_key, private_key = paillier.generate_paillier_keypair()
print('Keys generated')

Keys generated


## Encrypt Model Weights (Client 1)

In [7]:
enc_weights = [public_key.encrypt(w) for w in weightsA]
enc_bias = public_key.encrypt(biasA)

print('Weights encrypted and sent securely')

Weights encrypted and sent securely


## Homomorphic Aggregation Simulation (Client 2)

In [8]:
# simulate update
update = np.random.normal(0,0.01,len(enc_weights))

enc_updated_weights = [enc_weights[i] + update[i] for i in range(len(enc_weights))]

print('Encrypted update performed using homomorphic property')

Encrypted update performed using homomorphic property


## Decrypt Updated Weights

In [9]:
dec_weights = np.array([private_key.decrypt(w) for w in enc_updated_weights])

print('Weights decrypted successfully')

Weights decrypted successfully


## Client 2 Fine-tuning using transferred knowledge

In [10]:
ftl_model = LogisticRegression(max_iter=500)
ftl_model.fit(XB,yB)

# inject transferred weights
ftl_model.coef_ = dec_weights.reshape(1,-1)

pred_ftl = ftl_model.predict(X_test)

acc_ftl = accuracy_score(y_test,pred_ftl)

print('FTL Accuracy:', acc_ftl)

FTL Accuracy: 0.8421052631578947


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


## Result Comparison

In [11]:
results = pd.DataFrame({
    'Method':['Isolated Training','Federated Transfer Learning (Encrypted)'],
    'Accuracy':[acc_iso,acc_ftl]
})

results

,Method,Accuracy
0,Isolated Training,0.956140
1,Federated Transfer Learning (Encrypted),0.842105
